# Personal Information
Name: **Alexandra Holíková**

StudentID: **14236788**

Email: [**alexandra.holikova@student.uva.nl**](alexandra.holikova@student.uva.nl)

Github link: https://github.com/alexandraholik/MSc_Thesis.git

In [1]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
INPUT_CSV  = 'data_new/english_hand_annotated.csv'   # REPLACE WITH YOUR PATH
OUTPUT_CSV = 'data_new/english_matched.csv'
MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'

In [3]:
df = pd.read_csv(INPUT_CSV)
df['full_text'] = df['full_text'].fillna('')
df['headline_quote'] = df['headline_quote'].fillna('')
print(f"Loaded {len(df)} articles")

Loaded 100 articles


In [4]:
DQUOTE_RE = re.compile(
    u'[\"\u201c]([^\"\u201c\u201d]{20,300})[\"\u201d]'
)
SQUOTE_RE = re.compile(
    u"(?:^|[\\s\\-\u2014\u2013:])"
    u"[\u2018']"
    u"((?:[^'\u2018\u2019]|\u2019(?=\\w)){20,300})"
    u"[\u2019']"
    u"(?=[\\s.,;:!?\\-\u2014\u2013]|$)"
)

ATTRIB_RE = re.compile(
    r'\b(said|says|told|asked|added|argued|claimed|explained|noted|warned|'
    r'insisted|suggested|recalled|admitted|declared|announced|commented|'
    r'replied|responded|continued|wrote|tweeted|stated|according)\b',
    re.IGNORECASE
)
MIN_WORDS = 4

def extract_body_quotes(text):
    """Extract speech quotes: 4+ words, attribution verb within 80 chars."""
    quotes = []
    for pattern in [DQUOTE_RE, SQUOTE_RE]:
        for m in pattern.finditer(text):
            q = m.group(1).strip()
            if len(q.split()) < MIN_WORDS:
                continue
            start = m.start()
            window = text[max(0, start - 80):m.end() + 80]
            if ATTRIB_RE.search(window):
                quotes.append(q)
    return quotes

def split_sentences(text):
    """Strip HTML and split into sentences."""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]

In [5]:
embedder = SentenceTransformer(MODEL_NAME)
print(f"Model loaded: {MODEL_NAME}")


Model loaded: paraphrase-multilingual-mpnet-base-v2


In [6]:
results = []

for i, (_, row) in enumerate(df.iterrows()):
    hq = str(row['headline_quote']).strip()
    ft = str(row['full_text'])

    # Extract body quotes and sentences
    body_quotes = extract_body_quotes(ft)
    sentences   = split_sentences(ft)

    # Encode headline quote once
    hq_emb = embedder.encode([hq], normalize_embeddings=True)

    # --- Quote-to-quote matching ---
    if body_quotes:
        bq_emb  = embedder.encode(body_quotes, normalize_embeddings=True, batch_size=64)
        bq_sims = cosine_similarity(hq_emb, bq_emb)[0]
        best_bq_idx  = int(np.argmax(bq_sims))
        best_bq_sim  = float(bq_sims[best_bq_idx])
        best_bq      = body_quotes[best_bq_idx]
        # margin to second-best
        if len(bq_sims) > 1:
            sorted_bq = np.sort(bq_sims)[::-1]
            bq_margin = float(sorted_bq[0] - sorted_bq[1])
        else:
            bq_margin = best_bq_sim
    else:
        best_bq     = ''
        best_bq_sim = 0.0
        bq_margin   = 0.0

    # --- Sentence-level matching ---
    if sentences:
        sent_emb  = embedder.encode(sentences, normalize_embeddings=True, batch_size=64)
        sent_sims = cosine_similarity(hq_emb, sent_emb)[0]
        best_sent_idx = int(np.argmax(sent_sims))
        best_sent_sim = float(sent_sims[best_sent_idx])
        best_sent     = sentences[best_sent_idx]
        prev_sent     = sentences[best_sent_idx - 1] if best_sent_idx > 0 else ''
        next_sent     = sentences[best_sent_idx + 1] if best_sent_idx < len(sentences) - 1 else ''
        # margin
        if len(sent_sims) > 1:
            sorted_sent = np.sort(sent_sims)[::-1]
            sent_margin = float(sorted_sent[0] - sorted_sent[1])
        else:
            sent_margin = best_sent_sim
    else:
        best_sent = prev_sent = next_sent = ''
        best_sent_sim = sent_margin = 0.0

    results.append(dict(
        body_quotes      = ' ||| '.join(body_quotes),
        n_body_quotes    = len(body_quotes),
        best_body_quote  = best_bq,
        best_quote_sim   = round(best_bq_sim, 4),
        quote_margin     = round(bq_margin, 4),
        best_body_sentence = best_sent,
        prev_sentence    = prev_sent,
        next_sentence    = next_sent,
        best_sim         = round(best_sent_sim, 4),
        sim_margin       = round(sent_margin, 4),
    ))

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(df)} done")

  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done


In [7]:
res_df = pd.DataFrame(results, index=df.index)
out    = pd.concat([df, res_df], axis=1)
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"\nSaved: {OUTPUT_CSV}")
print(f"Rows with body quotes: {(out['n_body_quotes'] > 0).sum()}/{len(out)}")
print(f"Mean best_quote_sim: {out['best_quote_sim'].mean():.3f}")
print(f"Mean best_sim (sentence): {out['best_sim'].mean():.3f}")


Saved: data_new/english_matched.csv
Rows with body quotes: 98/100
Mean best_quote_sim: 0.743
Mean best_sim (sentence): 0.700


In [8]:
df = pd.read_csv('data_new/english_matched.csv')
no_bq = df[df['n_body_quotes'] == 0][['article_id', 'headline_quote', 'label']]
print(no_bq)

   article_id                                     headline_quote  label
7          73                     My wife belongs in the kitchen      1
8         960  I Almost Died Of Prostate Cancer And Painful E...      1


In [9]:
pos = df[df['label'] == 1][['article_id', 'headline_quote', 'best_body_quote', 'best_quote_sim']]
for _, r in pos.iterrows():
    print(f"ID {r['article_id']}")
    print(f"  HQ : {r['headline_quote'][:70]}")
    print(f"  BQ : {r['best_body_quote'][:70]}")
    print(f"  sim: {r['best_quote_sim']}")
    print()

ID 967
  HQ : We can sell anything, from your gumboots to your farm
  BQ : They can sell almost anything a farmer needs, from livestock, implemen
  sim: 0.7406

ID 1124
  HQ : wanted to fire candidates based on their looks
  BQ : All firing decisions throughout The Apprentice are down to Lord Sugar’
  sim: 0.4242

ID 389
  HQ : We need to burn down Arab villages - this is war!
  BQ : Just burn Arab villages in revenge; this is war,
  sim: 0.8812

ID 1326
  HQ : most dangerous president of all time
  BQ : I didn’t vote for him. If it’s going to get bad, [Donald Trump] could 
  sim: 0.6629

ID 920
  HQ : Those who wanted us out are disappointed
  BQ : Some people who wanted to discourage us, who wanted us out of the race
  sim: 0.6856

ID 1103
  HQ : I thought they would be safe
  BQ : I felt they would be as safe with him as they would be with me,
  sim: 0.7388

ID 2284
  HQ : Birth control is a political act
  BQ : Contraception is a feminist issue,
  sim: 0.7646

ID 73
  HQ : My wife 

TypeError: 'float' object is not subscriptable

In [10]:
import pandas as pd

df = pd.read_csv('data_new/english_matched.csv')

#ID 73 - Buhari: source is reporter narration, no extractable quote
#"He responded by reminding Nigeria's first lady where her place was: in his kitchen, his living room and 'the other room'"
df.loc[df['article_id'] == 73, 'best_body_quote'] = \
    'in his kitchen, his living room and "the other room"'
df.loc[df['article_id'] == 73, 'best_quote_sim'] = 0.0
df.loc[df['article_id'] == 73, 'n_body_quotes'] = 1
df.loc[df['article_id'] == 73, 'quote_margin'] = 0.0

#ID 960 - Prostate cancer: first-person prose, no quotation marks
#Source: "I would like to share my story...how I almost died from prostate cancer"
df.loc[df['article_id'] == 960, 'best_body_quote'] = \
    'I would like to share my story with you as regards how I almost died from prostate cancer.'
df.loc[df['article_id'] == 960, 'best_quote_sim'] = 0.0
df.loc[df['article_id'] == 960, 'n_body_quotes'] = 1
df.loc[df['article_id'] == 960, 'quote_margin'] = 0.0

df.loc[df['article_id'] == 2284, 'best_body_quote'] = \
    "Making my own choice about what my body can and cannot do in the face of an administration that wants to change that is a political act."
df.loc[df['article_id'] == 2284, 'best_quote_sim'] = 0.7646  

df.loc[df['article_id'] == 1124, 'best_body_quote'] = \
    "Do you think you could find it in your heart to not get rid of the blonde just yet?"


df.to_csv('data_new/english_matched.csv', index=False, encoding='utf-8-sig')

# Verify
print("Post-patch check on 10 positives:")
pos = df[df['label'] == 1][['article_id', 'headline_quote', 'best_body_quote',
                             'best_quote_sim', 'n_body_quotes']]
for _, r in pos.iterrows():
    print(f"\nID {r['article_id']} | n_bq={r['n_body_quotes']}")
    print(f"  HQ : {r['headline_quote'][:70]}")
    print(f"  BQ : {str(r['best_body_quote'])[:70]}")
    print(f"  sim: {r['best_quote_sim']}")

print(f"\nRows with n_body_quotes == 0: {(df['n_body_quotes'] == 0).sum()}")

Post-patch check on 10 positives:

ID 967 | n_bq=4
  HQ : We can sell anything, from your gumboots to your farm
  BQ : They can sell almost anything a farmer needs, from livestock, implemen
  sim: 0.7406

ID 1124 | n_bq=5
  HQ : wanted to fire candidates based on their looks
  BQ : Do you think you could find it in your heart to not get rid of the blo
  sim: 0.4242

ID 389 | n_bq=4
  HQ : We need to burn down Arab villages - this is war!
  BQ : Just burn Arab villages in revenge; this is war,
  sim: 0.8812

ID 1326 | n_bq=8
  HQ : most dangerous president of all time
  BQ : I didn’t vote for him. If it’s going to get bad, [Donald Trump] could 
  sim: 0.6629

ID 920 | n_bq=2
  HQ : Those who wanted us out are disappointed
  BQ : Some people who wanted to discourage us, who wanted us out of the race
  sim: 0.6856

ID 1103 | n_bq=7
  HQ : I thought they would be safe
  BQ : I felt they would be as safe with him as they would be with me,
  sim: 0.7388

ID 2284 | n_bq=14
  HQ : Birth contro